# Setting up a Simulation

First, we import the modules we need:

In [6]:
from ase.io import read
from ase.visualize import view

### Choosing the Starting structure

Now we load the structure that we previously created:

In [7]:
atoms = read("03_structure.cif")

Quick check if the structure looks correct:

In [8]:
view(atoms)

<Popen: returncode: None args: ['/scratch/data/marion/miniforge3/envs/ml/bin...>

### Choosing the Model

Now that we got our initial Structure we can choose a model that suits our use case. There are different models available.

Pick a machine‑learned potential suited to your system:

- Universal MOFs and general solids: https://github.com/ACEsuit/mace-foundations (MACE‑MP‑0b)
- MOF‑specific: MACE‑MP‑MOFv2 ()
- Small molecules: MACE‑OFF (for organics; not suited for MOFs) ()
- Alternative approach: UMA (different methodology) ()


MACE Foundations (models and docs): https://github.com/ACEsuit/mace-foundations

For this workshop we want to use a MACE model that suits MOFs.

In [11]:
from mace.calculators import mace

/Users/marionsappl/miniforge3/envs/ml/lib/python3.14/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [14]:
model = "models/mace-mh-1.model"
default_dtype = "float32"
calc = mace.MACECalculator(model_paths=model, default_dtype=default_dtype ,device="cpu", head="OMAT")

/Users/marionsappl/miniforge3/envs/ml/lib/python3.14/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [15]:
system.set_calculator(calc)

# Test energy/forces
e = system.get_potential_energy()
f = system.get_forces()
print("Initial energy (eV):", e)
print("Max force (eV/Å):", abs(f).max())

/var/folders/0j/30bqssmx3fj6z69n_h5ldpr00000gn/T/ipykernel_81492/1713781740.py:1: FutureWarning: Please use atoms.calc = calc
  system.set_calculator(calc)


Initial energy (eV): -862.9847412109375
Max force (eV/Å): 4.1592236


In [19]:
from ase.optimize import FIRE, BFGS

In [20]:
opt = BFGS(system, logfile=None)
opt.run(fmax=0.05)  # relax until max force <= 0.05 eV/Å

np.True_

In [21]:
view(system)

<Popen: returncode: None args: ['/Users/marionsappl/miniforge3/envs/ml/bin/p...>

In [22]:
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.optimize import BFGS
from ase.constraints import FixAtoms
from ase.io.trajectory import Trajectory
from ase.md import MDLogger
from ase import units
import numpy as np
import os

In [23]:
# 1) Initialize velocities and remove net momentum
MaxwellBoltzmannDistribution(system, temperature_K=300.0, rng=np.random.default_rng(42))
Stationary(system)

dt = 0.25 * units.fs
fric = 0.05
dyn = Langevin(system, dt, temperature_K=300.0, friction=fric)

report_interval = 100

steps = 1000

# 3) Clean observers, attach logger once
dyn.observers = []
os.makedirs("traj", exist_ok=True)
os.makedirs("out", exist_ok=True)

with Trajectory("traj/03_equilibration.traj", "w", system) as traj:
    dyn.attach(traj.write, interval=report_interval)

    def print_status():
        ekin = system.get_kinetic_energy()
        epot = system.get_potential_energy()
        N = len(system)
        T = 2.0 * ekin / (3.0 * N * units.kB)  # adjust ndof if you have constraints
        print(f"step={dyn.get_number_of_steps():4d}  Epot={epot:10.3f} eV  "
              f"Ekin={ekin:10.3f} eV  T={T:7.1f} K")
    dyn.attach(print_status, interval=report_interval)

    dyn.run(steps)

/var/folders/0j/30bqssmx3fj6z69n_h5ldpr00000gn/T/ipykernel_81492/2454437739.py:2: DeprecationWarning: Use thermalize_momenta
  MaxwellBoltzmannDistribution(system, temperature_K=300.0, rng=np.random.default_rng(42))
/Users/marionsappl/miniforge3/envs/ml/lib/python3.14/site-packages/ase/md/langevin.py:102: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


step=   0  Epot=  -866.548 eV  Ekin=     4.092 eV  T=  270.6 K
step= 100  Epot=  -864.282 eV  Ekin=     2.316 eV  T=  153.1 K
step= 200  Epot=  -863.696 eV  Ekin=     2.648 eV  T=  175.1 K
step= 300  Epot=  -864.148 eV  Ekin=     3.260 eV  T=  215.6 K
step= 400  Epot=  -863.780 eV  Ekin=     3.305 eV  T=  218.5 K
step= 500  Epot=  -863.187 eV  Ekin=     3.490 eV  T=  230.8 K
step= 600  Epot=  -862.996 eV  Ekin=     3.676 eV  T=  243.0 K
step= 700  Epot=  -862.660 eV  Ekin=     3.588 eV  T=  237.2 K
step= 800  Epot=  -862.410 eV  Ekin=     3.468 eV  T=  229.3 K
step= 900  Epot=  -862.475 eV  Ekin=     3.411 eV  T=  225.5 K
step=1000  Epot=  -862.690 eV  Ekin=     3.993 eV  T=  264.0 K
